In [ ]:
import numpy as np 


In [ ]:
tiflist = ['/scratch2/jungkyoj/comm_real/pair_20250919-20251001_morocco/frame_0_unwrapped_re.tif',
           '/scratch2/jungkyoj/comm_real/pair_20250919-20251001_morocco/frame_1_unwrapped_re.tif',
           '/scratch2/jungkyoj/comm_real/pair_20250919-20251001_morocco/frame_2_unwrapped_re.tif',
           '/scratch2/jungkyoj/comm_real/pair_20250919-20251001_morocco/frame_3_unwrapped_re.tif',
           '/scratch2/jungkyoj/comm_real/pair_20250919-20251001_morocco/frame_4_unwrapped_re.tif',
           '/scratch2/jungkyoj/comm_real/pair_20250919-20251001_morocco/frame_5_unwrapped_re.tif',
           '/scratch2/jungkyoj/comm_real/pair_20250919-20251001_morocco/frame_6_unwrapped_re.tif',]
           

In [ ]:
import itertools
from pathlib import Path

import numpy as np
import rasterio
from rasterio.windows import from_bounds
from rasterio.warp import reproject, Resampling
from rasterio.transform import array_bounds
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from scipy.ndimage import distance_transform_edt  # << new

tiflist = ['/scratch2/jungkyoj/comm_real/pair_20250919-20251001_morocco/frame_0_unwrapped_re.tif',
           '/scratch2/jungkyoj/comm_real/pair_20250919-20251001_morocco/frame_1_unwrapped_re.tif',
           '/scratch2/jungkyoj/comm_real/pair_20250919-20251001_morocco/frame_2_unwrapped_re.tif',
           '/scratch2/jungkyoj/comm_real/pair_20250919-20251001_morocco/frame_4_unwrapped_re.tif',]

OUT_DIR = Path("./overlap_diffs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BORDER_EROSION_PX = 10  # << shrink overlap by this many pixels from boundaries

def intersect_bounds(b1, b2):
    left = max(b1.left, b2.left)
    bottom = max(b1.bottom, b2.bottom)
    right = min(b1.right, b2.right)
    top = min(b1.top, b2.top)
    if left >= right or bottom >= top:
        return None
    from rasterio.coords import BoundingBox
    return BoundingBox(left, bottom, right, top)

def pairwise_overlap_diff(path_a, path_b, resampling=Resampling.bilinear, border_erosion_px=BORDER_EROSION_PX):
    """Return overlap-diff 'v' (A - B) and metadata for plotting, with boundary erosion."""
    with rasterio.open(path_a) as A, rasterio.open(path_b) as B:
        ib = intersect_bounds(A.bounds, B.bounds)
        if ib is None:
            return None

        winA = from_bounds(ib.left, ib.bottom, ib.right, ib.top, transform=A.transform)
        from math import floor
        from rasterio.windows import Window, transform as win_transform
        winA = Window(floor(winA.col_off), floor(winA.row_off), floor(winA.width), floor(winA.height))
        if winA.width <= 0 or winA.height <= 0:
            return None

        H, W = int(winA.height), int(winA.width)
        T_overlap = win_transform(winA, A.transform)

        a = A.read(1, window=winA).astype(np.float32)
        a_mask = np.zeros_like(a, dtype=bool)
        if A.nodata is not None:
            a_mask |= (a == A.nodata)

        b_reproj = np.empty((H, W), dtype=np.float32)
        b_reproj[:] = np.nan
        reproject(
            source=rasterio.band(B, 1),
            destination=b_reproj,
            src_transform=B.transform,
            src_crs=B.crs,
            src_nodata=B.nodata,
            dst_transform=T_overlap,
            dst_crs=A.crs,
            dst_nodata=np.nan,
            resampling=resampling
        )
        b_mask = np.isnan(b_reproj)

        valid = (~a_mask) & (~b_mask)

        # ----- NEW: erode the valid region by N pixels to remove boundary effects -----
        if border_erosion_px and border_erosion_px > 0:
            # Euclidean distance to the nearest invalid pixel; keep only interior > N
            dist = distance_transform_edt(valid)
            valid = dist > border_erosion_px
        # ------------------------------------------------------------------------------

        if not np.any(valid):
            return None

        v = a - b_reproj
        return {
            "A": path_a,
            "B": path_b,
            "v": v,
            "valid": valid,
            "transform": T_overlap,
            "crs": A.crs
        }

def robust_limits(x, lower=2, upper=98):
    lo = float(np.nanpercentile(x, lower))
    hi = float(np.nanpercentile(x, upper))
    return lo, hi

def stats_dict(vals):
    vals = np.asarray(vals)
    return {
        "count": int(vals.size),
        "mean": float(np.nanmean(vals)),
        "median": float(np.nanmedian(vals)),
        "std": float(np.nanstd(vals)),
        "MAE": float(np.nanmean(np.abs(vals))),
        "RMSE": float(np.sqrt(np.nanmean(vals**2))),
        "p05": float(np.nanpercentile(vals, 5)),
        "p95": float(np.nanpercentile(vals, 95)),
    }

def plot_hist_with_stats(vals, title):
    vals = np.asarray(vals)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        print(f"[skip] No finite values for {title}")
        return
    lo, hi = robust_limits(vals)
    import matplotlib.pyplot as plt
    from matplotlib.gridspec import GridSpec

    fig = plt.figure(figsize=(9, 4.2))
    gs = GridSpec(1, 2, width_ratios=[3, 1], figure=fig)
    ax_h = fig.add_subplot(gs[0, 0])
    ax_t = fig.add_subplot(gs[0, 1]); ax_t.axis("off")

    ax_h.hist(vals[(vals >= lo) & (vals <= hi)], bins=100)
    ax_h.set_title(title); ax_h.set_xlabel("A - B"); ax_h.set_ylabel("Count")

    s = stats_dict(vals)
    txt = (
        f"N: {s['count']}\n"
        f"Mean: {s['mean']:.4f}\n"
        f"Median: {s['median']:.4f}\n"
        f"Std: {s['std']:.4f}\n"
        f"MAE: {s['MAE']:.4f}\n"
        f"RMSE: {s['RMSE']:.4f}\n"
        f"P5–P95: {s['p05']:.4f} … {s['p95']:.4f}"
    )
    ax_t.text(0.02, 0.98, txt, va="top", ha="left", fontsize=11)
    fig.tight_layout(); plt.show()

def show_v_image(v, valid, transform, title="Overlap diff (A - B)"):
    v_plot = np.full_like(v, np.nan, dtype=np.float32)
    v_plot[valid] = v[valid]
    H, W = v_plot.shape
    y_min, y_max, x_min, x_max = array_bounds(H, W, transform)

    # symmetric robust color limits around zero
    abs99 = float(np.nanpercentile(np.abs(v_plot), 99))
    clim = abs99 if np.isfinite(abs99) and abs99 > 0 else np.nanmax(np.abs(v_plot))
    vmin, vmax = -clim, clim

    plt.figure(figsize=(6.6, 5.0))
    im = plt.imshow(v_plot, extent=(x_min, x_max, y_min, y_max), origin="upper", vmin=vmin, vmax=vmax)
    plt.title(title); plt.xlabel("X"); plt.ylabel("Y")
    plt.colorbar(im, fraction=0.046, pad=0.04, label="A - B")
    plt.tight_layout(); plt.show()

# ---------------- Run & plot ----------------
pair_results = []
all_vals = []

for pa, pb in itertools.combinations(tiflist, 2):
    res = pairwise_overlap_diff(pa, pb, resampling=Resampling.bilinear, border_erosion_px=BORDER_EROSION_PX)
    if res is None:
        print(f"[info] No (post-erosion) overlap for: {Path(pa).name} vs {Path(pb).name}")
        continue
    pair_results.append(res)
    v_valid = res["v"][res["valid"]]
    all_vals.append(v_valid)

    plot_hist_with_stats(v_valid, f"Histogram of diff (A-B): {Path(pa).name} vs {Path(pb).name}")
    show_v_image(res["v"], res["valid"], res["transform"], title=f"Overlap diff (A-B): {Path(pa).name} vs {Path(pb).name}")

if len(all_vals) > 0:
    agg = np.concatenate(all_vals)
    plot_hist_with_stats(agg, "Aggregated histogram of diff (A-B) across all overlaps (post-erosion)")
else:
    print("[info] No overlaps found after erosion.")
